In [1]:
import tensorflow as tf

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", len(gpus) > 0)
print("GPUs:", gpus)

# Print GPU details if available
if gpus:
    for i, gpu in enumerate(gpus):
        details = tf.config.experimental.get_device_details(gpu)
        print(f"GPU {i} Name:", details.get("device_name", "Unknown"))

GPU Available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU 0 Name: NVIDIA GeForce GTX 1650


In [2]:
print("Setting up Phase 5 training environment...")

# Import required libraries
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.metrics import Precision, Recall
import numpy as np
import time

# Architecture parameters
max_word_length = 75
max_char_length = 300
char_vocab_size = 72
char_embedding_dim = 50
struct_feature_dim = 135
filter_sizes = [3, 4, 5]
num_filters = 128
dropout_rate = 0.5

# Define model architecture functions (copied from Phase 4)
def create_word_branch_corrected(embedding_matrix):
    word_input = Input(shape=(max_word_length,), name='word_input')
    word_embedding = Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        input_length=max_word_length,
        trainable=True,
        name='word_embedding'
    )(word_input)
    word_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_filters, filter_size, activation='relu')(word_embedding)
        pool = MaxPooling1D(pool_size=max_word_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        word_convs.append(flatten)
    word_output = concatenate(word_convs)
    return word_input, word_output

def create_char_branch():
    char_input = Input(shape=(max_char_length,), name='char_input')
    char_embedding = Embedding(
        input_dim=char_vocab_size,
        output_dim=char_embedding_dim,
        input_length=max_char_length,
        trainable=True,
        name='char_embedding'
    )(char_input)
    char_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_filters, filter_size, activation='relu')(char_embedding)
        pool = MaxPooling1D(pool_size=max_char_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        char_convs.append(flatten)
    char_output = concatenate(char_convs)
    return char_input, char_output

def create_structural_branch():
    struct_input = Input(shape=(struct_feature_dim,), name='struct_input')
    struct_dense1 = Dense(256, activation='relu')(struct_input)
    struct_dropout1 = Dropout(dropout_rate)(struct_dense1)
    struct_dense2 = Dense(128, activation='relu')(struct_dropout1)
    return struct_input, struct_dense2

def build_multi_input_cnn_corrected(embedding_matrix):
    word_input, word_output = create_word_branch_corrected(embedding_matrix)
    char_input, char_output = create_char_branch()
    struct_input, struct_output = create_structural_branch()
    combined = concatenate([word_output, char_output, struct_output])
    combined = BatchNormalization()(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-5))(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-5))(combined)
    output = Dense(1, activation='sigmoid')(combined)
    model = Model(inputs=[word_input, char_input, struct_input], outputs=output)
    return model

# Load required data
print("Loading training data...")
embedding_matrix = np.load('embedding_matrix.npy')
X_train_words = np.load('X_train_preprocessed.npy')
X_train_chars = np.load('train_char_sequences.npy')
X_train_struct = np.load('train_sql_features.npy')
y_train = np.load('y_train_labels.npy')

X_val_words = np.load('X_val_preprocessed.npy')
X_val_chars = np.load('val_char_sequences.npy')
X_val_struct = np.load('val_sql_features.npy')
y_val = np.load('y_val_labels.npy')

print("Setup complete! Ready for Phase 5 training.")
print(f"Training samples: {X_train_words.shape[0]}")
print(f"Validation samples: {X_val_words.shape[0]}")


Setting up Phase 5 training environment...
Loading training data...
Setup complete! Ready for Phase 5 training.
Training samples: 55658
Validation samples: 11979


In [3]:
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import time

print("Building and training the final CNN model...")

# Define class weights from Phase 1
class_weights = {
    0: 1.4805809746754628,
    1: 0.7549508979436819
}

# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)

model_checkpoint = ModelCheckpoint(
    'best_model_phase5.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1,
    mode='min'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    mode='min',
    min_lr=1e-6
)

callbacks_list = [early_stopping, model_checkpoint, reduce_lr]

# Build the model using your best architecture
model = build_multi_input_cnn_corrected(embedding_matrix)

# Compile with best optimizer from Phase 4 (RMSprop)
model.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)

print("Model compiled successfully")
print(f"Total model parameters: {model.count_params():,}")

# Prepare training and validation data
train_inputs = [X_train_words, X_train_chars, X_train_struct]
train_labels = y_train
val_inputs = [X_val_words, X_val_chars, X_val_struct]
val_labels = y_val

print(f"Training data shape: {[x.shape for x in train_inputs]} + labels: {train_labels.shape}")
print(f"Validation data shape: {[x.shape for x in val_inputs]} + labels: {val_labels.shape}")

print("Training configuration:")
print(f"- Optimizer: RMSprop (lr=0.001)")
print(f"- Batch size: 32")
print(f"- Max epochs: 25")
print(f"- Early stopping patience: 7")
print(f"- Class weights: {class_weights}")
print("- Callbacks: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau")

# Train the model with class weights
print("\nStarting model training...")
print("=" * 60)

start_time = time.time()

history = model.fit(
    x=train_inputs,
    y=train_labels,
    batch_size=32,
    epochs=25,
    validation_data=(val_inputs, val_labels),
    class_weight=class_weights,
    callbacks=callbacks_list,
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time

print("=" * 60)
print("Model training completed!")
print(f"Total training time: {training_time//60:.0f}m {training_time%60:.0f}s")
print(f"Final epoch reached: {len(history.history['loss'])}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")


Building and training the final CNN model...
Model compiled successfully
Total model parameters: 6,859,409
Training data shape: [(55658, 75), (55658, 300), (55658, 135)] + labels: (55658,)
Validation data shape: [(11979, 75), (11979, 300), (11979, 135)] + labels: (11979,)
Training configuration:
- Optimizer: RMSprop (lr=0.001)
- Batch size: 32
- Max epochs: 25
- Early stopping patience: 7
- Class weights: {0: 1.4805809746754628, 1: 0.7549508979436819}
- Callbacks: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

Starting model training...
Epoch 1/25
1740/1740 [==============================] - ETA: 0s - loss: 0.0939 - accuracy: 0.9730 - precision: 0.9951 - recall: 0.9641
Epoch 1: val_loss improved from inf to 0.06083, saving model to best_model_phase5.h5
1740/1740 [==============================] - 62s 31ms/step - loss: 0.0939 - accuracy: 0.9730 - precision: 0.9951 - recall: 0.9641 - val_loss: 0.0608 - val_accuracy: 0.9812 - val_precision: 0.9996 - val_recall: 0.9718 - lr: 0.0010
Epo